# Family Fuel Dashboard — Exploratory Data Analysis

This notebook is a scratchpad for exploring the data after an ETL run.
It queries DuckDB directly and visualizes what's in the warehouse.

**Run the ETL first:**
```bash
python etl/run_pipeline.py
```

In [ ]:
import sys
sys.path.insert(0, '..')

import duckdb
import pandas as pd

con = duckdb.connect('../data/family_fuel.duckdb')
print('Connected to DuckDB')

## 1. What's in the database?

In [ ]:
# Row counts across all tables
tables = ['raw_recipes', 'raw_usda_nutrients', 'raw_off_products',
          'dim_recipes', 'dim_ingredients', 'fact_recipe_nutrition',
          'fact_recipe_ingredients', 'fact_meal_plans']

for t in tables:
    try:
        n = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
        print(f'{t:<30} {n:>6} rows')
    except Exception as e:
        print(f'{t:<30} ERROR: {e}')

## 2. Recipe Library Overview

In [ ]:
recipes = con.execute("""
    SELECT
        title,
        cuisine,
        prep_minutes,
        servings,
        kid_friendly,
        diet_tags
    FROM dim_recipes
    ORDER BY prep_minutes
    LIMIT 20
""").df()
recipes

In [ ]:
# Prep time distribution
prep_dist = con.execute("""
    SELECT
        CASE
            WHEN prep_minutes <= 20 THEN '0-20 min'
            WHEN prep_minutes <= 30 THEN '21-30 min'
            WHEN prep_minutes <= 45 THEN '31-45 min'
            ELSE '45+ min'
        END AS prep_bucket,
        COUNT(*) AS recipe_count
    FROM dim_recipes
    WHERE prep_minutes IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").df()
prep_dist

## 3. Nutrition Analysis

In [ ]:
# Average macros per recipe
macros = con.execute("""
    SELECT
        dn.name AS nutrient,
        dn.unit,
        ROUND(AVG(frn.amount_per_serving), 1) AS avg_per_serving,
        ROUND(MIN(frn.amount_per_serving), 1) AS min_per_serving,
        ROUND(MAX(frn.amount_per_serving), 1) AS max_per_serving
    FROM fact_recipe_nutrition frn
    JOIN dim_nutrients dn USING (nutrient_id)
    GROUP BY dn.name, dn.unit
    ORDER BY dn.nutrient_id
""").df()
macros

In [ ]:
# Top 10 highest-protein recipes
con.execute("""
    SELECT
        dr.title,
        dr.prep_minutes,
        ROUND(frn.amount_per_serving, 1) AS protein_g
    FROM fact_recipe_nutrition frn
    JOIN dim_recipes dr USING (recipe_id)
    JOIN dim_nutrients dn USING (nutrient_id)
    WHERE dn.name = 'Protein'
    ORDER BY frn.amount_per_serving DESC
    LIMIT 10
""").df()

## 4. This Week's Meal Plan

In [ ]:
# Load the mart_weekly_plans view
plan = con.execute("""
    SELECT
        day_of_week,
        meal_slot,
        title,
        prep_minutes,
        ROUND(calories_per_serving) AS calories
    FROM mart_weekly_plans
    ORDER BY day_of_week, meal_slot
""").df()

# Pivot for a nice week view
day_names = {1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat', 7: 'Sun'}
plan['day'] = plan['day_of_week'].map(day_names)
plan.pivot(index='meal_slot', columns='day', values='title')

## 5. Weekly Macro Totals

In [ ]:
con.execute("""
    SELECT
        day_of_week,
        ROUND(total_calories) AS calories,
        ROUND(total_protein_g, 1) AS protein_g,
        ROUND(total_carbs_g, 1) AS carbs_g,
        ROUND(total_fat_g, 1) AS fat_g
    FROM mart_macro_summary
""").df()

## 6. Pipeline Health

In [ ]:
con.execute("""
    SELECT
        source,
        status,
        rows_loaded,
        run_at,
        error_message
    FROM raw_pipeline_runs
    ORDER BY run_at DESC
    LIMIT 20
""").df()